# Task 2: Quantitative analysis using pynance and TaLib 
Objective: Load historical stock price data, compute financial technical indicators, and visualize the results to understand market behavior. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import ta                        # technical analysis library
import warnings
warnings.filterwarnings('ignore')

### preparing the data
a. Load the stock price dataset into a pandas DataFrame  
b. Ensure columns are correctly typed (Open, High, Low, Close, Volume)  
c. Check for and handle missing values 

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

In [ ]:
TICKER = 'AAPL'
#pick 4 year data for training and testing
START   = '2019-01-01'
END     = '2023-12-31'

DATA_PATH = '../data/raw/AAPL.csv'
df = pd.read_csv(DATA_PATH, index_col=0)

raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)

print(f'Downloaded shape: {raw.shape}')
print(f'Columns (first level): {raw.columns.get_level_values(0).unique().tolist()}')
df.head(3)

# Data cleaning and quality check

## Technical Indicators with `ta` Library

Technical indicators are mathematical transformations of price/volume data used to identify trends, momentum, and potential reversals.


### Moving Averages (SMA & EMA)

**Simple Moving Average (SMA):**  
Average closing price over the last 20,50 etc days. All days weighted equally.

**Exponential Moving Average (EMA):**  
Like SMA but more recent days get higher weight. Reacts faster to price changes.  
Common windows: 20 days (short-term), 50 days (medium), 200 days (long-term trend).

In [ ]:
# SMA
# # ta.trend.sma_indicator(series, window) returns a pandas Series of SMA values.
# The first (window-1) rows will be NaN because there isn't enough history yet.
df['SMA_20']  = ta.trend.sma_indicator(df['Close'], window=20)
df['SMA_50']  = ta.trend.sma_indicator(df['Close'], window=50)
df['SMA_200'] = ta.trend.sma_indicator(df['Close'], window=200)

# EMA 
df['EMA_12'] = ta.trend.ema_indicator(df['Close'], window=12)
df['EMA_26'] = ta.trend.ema_indicator(df['Close'], window=26)

print('Moving averages computed')
df[['Close','SMA_20','SMA_50','EMA_12']].tail(5).round(2)

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(df.index, df['Close'], label='Close Price', linewidth=2)

plt.plot(df.index, df['SMA_20'], label='SMA 20', linestyle='--')
plt.plot(df.index, df['SMA_50'], label='SMA 50', linestyle='--')

plt.plot(df.index, df['EMA_12'], label='EMA 12')
plt.plot(df.index, df['EMA_26'], label='EMA 26')

plt.title('Stock Price with SMA and EMA Indicators')
plt.xlabel('Date')
plt.ylabel('Price')

plt.legend()

plt.tight_layout()
plt.show()

# Relative Strength Index

In [ ]:
# RSIIndicator takes the Close series and the lookback window (default 14).
rsi_indicator = ta.momentum.RSIIndicator(close=df['Close'], window=14)
df['RSI_14'] = rsi_indicator.rsi()

print('RSI computed')
print(f'RSI range: {df["RSI_14"].min():.1f} – {df["RSI_14"].max():.1f}')
df[['Close','RSI_14']].tail(5).round(2)
print(df.columns)

In [ ]:
plt.figure(figsize=(14,4))

plt.plot(df.index, df['RSI_14'], label='RSI 14')

plt.axhline(70, linestyle='--', label='Overbought (70)')
plt.axhline(30, linestyle='--', label='Oversold (30)')

plt.title('Relative Strength Index (RSI)')
plt.xlabel('Date')
plt.ylabel('RSI')

plt.legend()

plt.tight_layout()
plt.show()

# Moving Average Convergence Divergence

In [ ]:
macd = ta.trend.MACD(close=df['Close'])

df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()
df['MACD_diff'] = macd.macd_diff()

print('MACD computed ✅')
df[['Close','MACD','MACD_Signal','MACD_Hist']].tail(5).round(4)

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(df.index, df['MACD'], label='MACD')
plt.plot(df.index, df['MACD_signal'], label='Signal Line')

plt.bar(df.index, df['MACD_diff'], label='Histogram')

plt.title('MACD Indicator')
plt.xlabel('Date')

plt.legend()

plt.tight_layout()
plt.show()